# coconut-audit — quickstart (demo mode)

End-to-end pipeline against the synthetic v0.1.0 demo path:
1. Build an `AuditConfig`.
2. Run the steering / shortcut / drift probes against synthetic activations.
3. Inspect the JSON report and the HTML rendering.

Real-model audits (HF inference + SAE encode + benchmark loop) ship in v0.1.1+.

In [ ]:
%pip install -q coconut-audit

In [ ]:
from pathlib import Path

from coconut_audit.core import AuditConfig
from coconut_audit.core.types import ProbeKind
from coconut_audit.mcp import run_audit_pipeline
from coconut_audit.reports import dump_json_report, render_html_report

out_dir = Path('audit_reports')
out_dir.mkdir(exist_ok=True)

for probe in (ProbeKind.STEERING, ProbeKind.SHORTCUT, ProbeKind.DRIFT):
    cfg = AuditConfig(
        model_id='gpt2',
        sae_id='jbloom/GPT2-Small-SAEs',
        probe_kind=probe,
        n_samples=8,
        seed=42,
    )
    report = run_audit_pipeline(cfg, demo_mode=True, ledger_path=out_dir / 'ledger.jsonl')
    dump_json_report(report, out_dir / f'{report.audit_id}.json')
    render_html_report(report, out_dir / f'{report.audit_id}.html')
    print(f'{probe.value:>9s}  verdict={report.verdict.value:<10s} id={report.audit_id}')

## MCP usage

Register the stdio server with Claude Code / Cursor / Cline:

```jsonc
// ~/.config/claude/mcp.json
{
  "mcpServers": {
    "coconut-audit": { "command": "coconut-audit-mcp" }
  }
}
```

Then call `audit_run`, `audit_get`, `audit_diff` from the agent.